In [3]:
import time
from pymongo import MongoClient

NOMBRE_GRUPO = "DataTraders"
CATEGORIA_ACTUAL = "Finanzas"

client = MongoClient("database", 27017)
db = client["proyecto_bigdata"]
coleccion = db["datos_finanzas"]

print("CONEXION EXITOSA:", CATEGORIA_ACTUAL)

CONEXION EXITOSA: Finanzas


In [4]:
lista_items = [
    {"titulo": "Apple Inc.", "precio": "189.45"},
    {"titulo": "Tesla Inc.", "precio": "172.30"},
    {"titulo": "Nvidia Corp.", "precio": "903.10"}
]

In [5]:
for item in lista_items:
    registro = {
        "item": item["titulo"],
        "valor": float(item["precio"].replace(" ", "").replace(",", "").strip()),
        "categoria": CATEGORIA_ACTUAL,
        "grupo": NOMBRE_GRUPO,
        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    coleccion.insert_one(registro)

print("PROCESO FINALIZADO: Datos enviados a MongoDB.")

PROCESO FINALIZADO: Datos enviados a MongoDB.


In [6]:
docs = list(coleccion.find())

print("Documentos encontrados:", len(docs))

for doc in docs[:5]:
    print(doc)

Documentos encontrados: 5
{'_id': ObjectId('69d653f9931c8aee880d28dd'), 'item': 'Apple Inc.', 'valor': 189.45, 'categoria': 'Finanzas', 'grupo': 'DataTraders', 'fecha_captura': '2026-04-08 13:11:21'}
{'_id': ObjectId('69d653f9931c8aee880d28de'), 'item': 'Tesla Inc.', 'valor': 172.3, 'categoria': 'Finanzas', 'grupo': 'DataTraders', 'fecha_captura': '2026-04-08 13:11:21'}
{'_id': ObjectId('69df0a7536be3a290e110ec7'), 'item': 'Apple Inc.', 'valor': 189.45, 'categoria': 'Finanzas', 'grupo': 'DataTraders', 'fecha_captura': '2026-04-15 03:48:05'}
{'_id': ObjectId('69df0a7536be3a290e110ec8'), 'item': 'Tesla Inc.', 'valor': 172.3, 'categoria': 'Finanzas', 'grupo': 'DataTraders', 'fecha_captura': '2026-04-15 03:48:05'}
{'_id': ObjectId('69df0a7536be3a290e110ec9'), 'item': 'Nvidia Corp.', 'valor': 903.1, 'categoria': 'Finanzas', 'grupo': 'DataTraders', 'fecha_captura': '2026-04-15 03:48:05'}


In [12]:
import pandas as pd

docs = list(coleccion.find())

df_pandas = pd.DataFrame(docs)

if "_id" in df_pandas.columns:
    df_pandas["_id"] = df_pandas["_id"].astype(str)

df_pandas.head()

,_id,item,valor,categoria,grupo,fecha_captura
0,69d653f9931c8aee880d28dd,Apple Inc.,189.45,Finanzas,DataTraders,2026-04-08 13:11:21
1,69d653f9931c8aee880d28de,Tesla Inc.,172.30,Finanzas,DataTraders,2026-04-08 13:11:21
2,69df0a7536be3a290e110ec7,Apple Inc.,189.45,Finanzas,DataTraders,2026-04-15 03:48:05
3,69df0a7536be3a290e110ec8,Tesla Inc.,172.30,Finanzas,DataTraders,2026-04-15 03:48:05
4,69df0a7536be3a290e110ec9,Nvidia Corp.,903.10,Finanzas,DataTraders,2026-04-15 03:48:05


In [14]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis_DataTraders") \
    .getOrCreate()

In [15]:
df = spark.createDataFrame(df_pandas)

df.show()

+--------------------+------------+------+---------+-----------+-------------------+
|                 _id|        item| valor|categoria|      grupo|      fecha_captura|
+--------------------+------------+------+---------+-----------+-------------------+
|69d653f9931c8aee8...|  Apple Inc.|189.45| Finanzas|DataTraders|2026-04-08 13:11:21|
|69d653f9931c8aee8...|  Tesla Inc.| 172.3| Finanzas|DataTraders|2026-04-08 13:11:21|
|69df0a7536be3a290...|  Apple Inc.|189.45| Finanzas|DataTraders|2026-04-15 03:48:05|
|69df0a7536be3a290...|  Tesla Inc.| 172.3| Finanzas|DataTraders|2026-04-15 03:48:05|
|69df0a7536be3a290...|Nvidia Corp.| 903.1| Finanzas|DataTraders|2026-04-15 03:48:05|
+--------------------+------------+------+---------+-----------+-------------------+



In [16]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

windowSpec = Window.partitionBy("item").orderBy(col("fecha_captura").desc())

df_unico = df.withColumn("rank", row_number().over(windowSpec)) \
             .filter(col("rank") == 1) \
             .drop("rank")

df_unico.show(truncate=False)

+------------------------+------------+------+---------+-----------+-------------------+
|_id                     |item        |valor |categoria|grupo      |fecha_captura      |
+------------------------+------------+------+---------+-----------+-------------------+
|69df0a7536be3a290e110ec7|Apple Inc.  |189.45|Finanzas |DataTraders|2026-04-15 03:48:05|
|69df0a7536be3a290e110ec9|Nvidia Corp.|903.1 |Finanzas |DataTraders|2026-04-15 03:48:05|
|69df0a7536be3a290e110ec8|Tesla Inc.  |172.3 |Finanzas |DataTraders|2026-04-15 03:48:05|
+------------------------+------------+------+---------+-----------+-------------------+



In [17]:
from pyspark.sql.functions import col

df_filtrado = df_unico.filter(
    (col("item").isNotNull()) & 
    (col("valor") > 0)
)

print("Registros originales:", df.count())
print("Registros válidos:", df_filtrado.count())

Registros originales: 5
Registros válidos: 3


In [18]:
df_filtrado.select("item", "valor").show(truncate=False)

df_filtrado.filter(col("valor") > 100).show(truncate=False)

df_filtrado.groupBy("grupo").count().show()

+------------+------+
|item        |valor |
+------------+------+
|Apple Inc.  |189.45|
|Nvidia Corp.|903.1 |
|Tesla Inc.  |172.3 |
+------------+------+

+------------------------+------------+------+---------+-----------+-------------------+
|_id                     |item        |valor |categoria|grupo      |fecha_captura      |
+------------------------+------------+------+---------+-----------+-------------------+
|69df0a7536be3a290e110ec7|Apple Inc.  |189.45|Finanzas |DataTraders|2026-04-15 03:48:05|
|69df0a7536be3a290e110ec9|Nvidia Corp.|903.1 |Finanzas |DataTraders|2026-04-15 03:48:05|
|69df0a7536be3a290e110ec8|Tesla Inc.  |172.3 |Finanzas |DataTraders|2026-04-15 03:48:05|
+------------------------+------------+------+---------+-----------+-------------------+

+-----------+-----+
|      grupo|count|
+-----------+-----+
|DataTraders|    3|
+-----------+-----+



In [19]:
from pyspark.sql import functions as F

reporte_categorias = df_filtrado.groupBy("categoria").agg(
    F.count("item").alias("cantidad_items"),
    F.avg("valor").alias("valor_promedio"),
    F.min("valor").alias("valor_minimo"),
    F.max("valor").alias("valor_maximo")
)

reporte_categorias.show()

+---------+--------------+-----------------+------------+------------+
|categoria|cantidad_items|   valor_promedio|valor_minimo|valor_maximo|
+---------+--------------+-----------------+------------+------------+
| Finanzas|             3|421.6166666666666|       172.3|       903.1|
+---------+--------------+-----------------+------------+------------+



In [20]:
ticket_salida = df_filtrado.filter(col("grupo") == "DataTraders") \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_items"),
        F.avg("valor").alias("valor_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

ticket_salida.show()

+---------+-----------+-----------------+---------------------+
|categoria|total_items|      valor_medio|ultima_sincronizacion|
+---------+-----------+-----------------+---------------------+
| Finanzas|          3|421.6166666666666|  2026-04-15 03:48:05|
+---------+-----------+-----------------+---------------------+

